# Eval Relaxed Cuts For fitstat0_2727

This notebook compares baseline vs relaxed evaluation cuts for the trained model `fitstat0_2727`.

Groups:

1. `dcedge`: baseline `dcedge_min=20`, relaxed `dcedge_min=0`
2. `dangle`: baseline `dangle_max_deg=3`, relaxed `dangle_max_deg=999`
3. `pincness`: baseline `pinc_max=1.1`, relaxed `pinc_max=9999`

Each group produces three comparison plots:

- `resolution_weighted`
- `logRMS_weighted`
- `bias_weighted`


## 1. Set paths and import shared utilities

Prepare the current project root, output directory, and the shared weighted-stat helpers from `src/common`.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/home/server/projects/energy_reconstruction')
RUN_DIR = PROJECT_ROOT / 'runs' / 'fitstat0_2727'
OUTPUT_DIR = PROJECT_ROOT / 'notebook' / 'generated' / 'eval_relaxed_cuts_fitstat0_2727'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.common.utils import _weighted_mean, _weighted_std, _weighted_rms

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print('RUN_DIR    =', RUN_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 2. Define expected evaluation directories

The notebook does not hard-code any metric numbers. It reads `effective_eval_config.json`, `metrics.json`, and `preds.npz` from the evaluation output directories.


In [ ]:
GROUPS = {
    'dcedge': {
        'baseline': 'fig_eval_dcedge20_baseline',
        'relaxed': 'fig_eval_dcedge0_relaxed',
        'baseline_label': 'Baseline: dcedge_min = 20',
        'relaxed_label': 'Relaxed: dcedge_min = 0',
    },
    'dangle': {
        'baseline': 'fig_eval_dangle3_baseline',
        'relaxed': 'fig_eval_dangle_relaxed',
        'baseline_label': 'Baseline: dangle_max_deg = 3',
        'relaxed_label': 'Relaxed: dangle_max_deg = 999',
    },
    'pincness': {
        'baseline': 'fig_eval_pinc1p1_baseline',
        'relaxed': 'fig_eval_pinc_relaxed',
        'baseline_label': 'Baseline: pinc_max = 1.1',
        'relaxed_label': 'Relaxed: pinc_max = 9999',
    },
}

RESULTS = {}
for group_name, item in GROUPS.items():
    RESULTS[group_name] = {}
    for mode in ['baseline', 'relaxed']:
        out_dir = RUN_DIR / item[mode]
        effective_config_path = out_dir / 'effective_eval_config.json'
        metrics_path = out_dir / 'metrics.json'
        preds_path = out_dir / 'preds.npz'

        assert out_dir.exists(), f'Missing evaluation dir: {out_dir}'
        assert effective_config_path.exists(), f'Missing effective config: {effective_config_path}'
        assert metrics_path.exists(), f'Missing metrics: {metrics_path}'
        assert preds_path.exists(), f'Missing preds: {preds_path}'

        RESULTS[group_name][mode] = {
            'out_dir': out_dir,
            'effective_config_path': effective_config_path,
            'metrics_path': metrics_path,
            'preds_path': preds_path,
            'display': item[f'{mode}_label'],
        }

for group_name, pair in RESULTS.items():
    print(group_name)
    for mode, item in pair.items():
        print(f'  {mode}: {item["out_dir"]}')


## 3. Load configs, metrics, and prediction arrays

For each baseline/relaxed pair, load the saved prediction arrays and metrics, then recompute the weighted curves using the same weighting definition as `src/common/utils.py`.


In [ ]:
loaded = {}
for group_name, pair in RESULTS.items():
    loaded[group_name] = {}
    for mode, item in pair.items():
        with open(item['effective_config_path'], 'r') as f:
            effective_config = json.load(f)
        with open(item['metrics_path'], 'r') as f:
            metrics = json.load(f)
        preds = np.load(item['preds_path'])

        loaded[group_name][mode] = {
            'display': item['display'],
            'effective_config': effective_config,
            'metrics': metrics,
            'logE_true': preds['logE_true'].astype(np.float64),
            'logE_pred': preds['logE_pred'].astype(np.float64),
            'mc_weight': preds['mc_weight'].astype(np.float64),
        }

for group_name, pair in loaded.items():
    print(group_name)
    for mode, item in pair.items():
        print({
            'mode': mode,
            'display': item['display'],
            'n': item['metrics'].get('n'),
            'w_log_sigma': item['metrics'].get('w_log_sigma'),
            'w_log_rmse': item['metrics'].get('w_log_rmse'),
            'w_log_bias': item['metrics'].get('w_log_bias'),
        })


## 4. Recompute weighted comparison curves with shared bins

For each group, use a shared true-energy binning across baseline and relaxed predictions so the two curves can be compared directly.


In [ ]:
def compute_weighted_curves(log_true, log_pred, weights, bin_edges):
    mask = (
        np.isfinite(log_true)
        & np.isfinite(log_pred)
        & np.isfinite(weights)
        & (weights > 0)
    )

    log_true = log_true[mask]
    log_pred = log_pred[mask]
    weights = weights[mask]

    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bias = []
    resolution = []
    log_rms = []

    for i in range(len(bin_edges) - 1):
        m = (log_true >= bin_edges[i]) & (log_true < bin_edges[i + 1])
        if m.sum() > 10:
            residual = log_pred[m] - log_true[m]
            w = weights[m]
            bias.append(float(_weighted_mean(residual, w)))
            resolution.append(float(_weighted_std(log_pred[m], w)))
            log_rms.append(float(_weighted_rms(residual, w)))
        else:
            bias.append(np.nan)
            resolution.append(np.nan)
            log_rms.append(np.nan)

    return {
        'bin_centers': centers,
        'bias': np.asarray(bias, dtype=np.float64),
        'resolution': np.asarray(resolution, dtype=np.float64),
        'log_rms': np.asarray(log_rms, dtype=np.float64),
    }

curves = {}
for group_name, pair in loaded.items():
    all_true = np.concatenate([
        pair['baseline']['logE_true'][np.isfinite(pair['baseline']['logE_true'])],
        pair['relaxed']['logE_true'][np.isfinite(pair['relaxed']['logE_true'])],
    ])
    bin_edges = np.linspace(all_true.min(), all_true.max(), 21)
    curves[group_name] = {
        mode: compute_weighted_curves(
            pair[mode]['logE_true'],
            pair[mode]['logE_pred'],
            pair[mode]['mc_weight'],
            bin_edges,
        )
        for mode in ['baseline', 'relaxed']
    }
    print(group_name, 'shared true-energy range =', (float(all_true.min()), float(all_true.max())))


## 5. Plot baseline vs relaxed for each variable

This cell creates 9 figures in total: 3 variables × 3 weighted metrics.


In [ ]:
MODE_STYLE = {
    'baseline': {'color': '#1f77b4', 'marker': 'o'},
    'relaxed': {'color': '#d62728', 'marker': 's'},
}

def plot_group_compare(group_name, metric_key, ylabel, title_suffix, filename_suffix, draw_zero=False):
    fig, ax = plt.subplots(figsize=(8, 5))
    for mode in ['baseline', 'relaxed']:
        item = loaded[group_name][mode]
        curve = curves[group_name][mode]
        style = MODE_STYLE[mode]
        ax.plot(
            curve['bin_centers'],
            curve[metric_key],
            marker=style['marker'],
            color=style['color'],
            linewidth=2,
            markersize=5,
            label=item['display'],
        )

    if draw_zero:
        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1)

    ax.set_xlabel(r'True Energy $\log_{10}(E/\mathrm{GeV})$')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{group_name}: {title_suffix}')
    ax.legend(frameon=True)
    ax.grid(alpha=0.3)
    fig.tight_layout()

    out_path = OUTPUT_DIR / f'{group_name}_{filename_suffix}.png'
    fig.savefig(out_path, dpi=200)
    plt.show()
    print('saved:', out_path)
    return out_path

generated_paths = []
for group_name in ['dcedge', 'dangle', 'pincness']:
    generated_paths.append(plot_group_compare(
        group_name=group_name,
        metric_key='resolution',
        ylabel=r'Weighted resolution of $\log_{10}(E_\mathrm{pred}/\mathrm{GeV})$',
        title_suffix='Weighted Resolution Comparison',
        filename_suffix='resolution_weighted_compare',
        draw_zero=False,
    ))
    generated_paths.append(plot_group_compare(
        group_name=group_name,
        metric_key='log_rms',
        ylabel=r'Weighted RMS of $\Delta\log_{10}E$',
        title_suffix='Weighted Log-RMS Comparison',
        filename_suffix='logRMS_weighted_compare',
        draw_zero=False,
    ))
    generated_paths.append(plot_group_compare(
        group_name=group_name,
        metric_key='bias',
        ylabel=r'Weighted bias of $\Delta\log_{10}E$',
        title_suffix='Weighted Bias Comparison',
        filename_suffix='bias_weighted_compare',
        draw_zero=True,
    ))

print('generated', len(generated_paths), 'figures')
